# INF201 Deliverable 3

**Group**: G4_1


## Task 1 Update Character



In [13]:
from game_simulator_v2 import Character, Warrior, Mage

hero = Character("Hero", hp_max=30, damage=8)
enemy = Character("Enemy", hp_max=20)

hero.introduce()
# Task 1: attack() uses the enemy's take_damage() method.
hero.attack(enemy)
print("Enemy HP after attack:", enemy.hp)

# Damage cannot lower HP below 0.
hero.take_damage(12)
print("Hero HP after damage:", hero.hp)

# Healing cannot exceed hp_max (30).
hero.heal(50)
print("Hero HP after healing:", hero.hp)

# is_alive is True when HP > 0.
print("Hero is alive:", hero.is_alive)

enemy.take_damage(100)
print("Enemy HP:", enemy.hp, "| Alive:", enemy.is_alive)

Hi, my name is Hero!
Enemy HP after attack: 12
Hero HP after damage: 18
Hero HP after healing: 30
Hero is alive: True
Enemy HP: 0 | Alive: False


## Task 2 Dice


In [14]:
%%writefile dice.py
import random


class Dice:
    def __init__(self, sides=6):
        # Any number of sides >= 2; default is 6.
        if sides < 2:
            raise ValueError("A dice must have at least 2 sides")
        self.sides = sides

    def roll(self):
        # Roll a random number from 1 to sides.
        return random.randint(1, self.sides)

    def __repr__(self):
        # Show the dice as Dice(sides=6).
        return f"Dice(sides={self.sides})"

    def __call__(self):
        # Allow d20() to roll the dice, like d20.roll().
        return self.roll()


Overwriting dice.py


In [15]:
from dice import Dice

# A default 6-sided dice and a custom 20-sided dice.
d6 = Dice()
d20 = Dice(20)

#  __repr__ prints Dice(sides=6), etc.
print("Dice:", d6, d20)
# roll() and __call__() both roll the dice.
print("Roll:", d6.roll())
print("Call:", d20())

Dice: Dice(sides=6) Dice(sides=20)
Roll: 6
Call: 17


## Task 3 Weapon



In [16]:
%%writefile items.py
from dice import Dice


class Weapon:
    def __init__(self, name: str, damage_dice: Dice):
        #  A weapon has a name and a Dice object.
        self.name = name
        self.damage_dice = damage_dice

    def roll_damage(self):
        #  Roll the weapon's dice to calculate damage.
        return self.damage_dice.roll()


Overwriting items.py


In [17]:
from items import Weapon

#  Weapon stores a name and a damage dice.
sword = Weapon("Sword", Dice(12))
print("Weapon:", sword.name)
print("Damage dice:", sword.damage_dice)

# Character accepts an optional weapon and rolls weapon damage.
fighter = Character("Fighter", hp_max=30, weapon=sword)
target = Character("Target", hp_max=30)
fighter.attack(target)
print("Target HP after weapon attack:", target.hp)

Weapon: Sword
Damage dice: Dice(sides=12)
Target HP after weapon attack: 28


## Task 4 Warrior and Mage



In [18]:
# Warrior armor reduces incoming damage.
warrior = Warrior("Warrior", hp_max=30, armor=5)
warrior.take_damage(12)
print("Warrior HP after 12 damage:", warrior.hp)
print("Warrior armor:", warrior.armor)

# Mage spends 10 mana on magic, then uses a normal weapon attack.
mage = Mage("Mage", damage=6, mana=20, mana_max=20, weapon=sword)
target = Character("Training dummy", hp_max=100)

for turn in range(3):
    mage.attack(target)
    print(f"Attack {turn + 1}: Target HP = {target.hp}, Mage mana = {mage.mana}")

Warrior HP after 12 damage: 23
Warrior armor: 5
Attack 1: Target HP = 88, Mage mana = 10
Attack 2: Target HP = 76, Mage mana = 0
Attack 3: Target HP = 72, Mage mana = 5


## Task 5 Battle



In [19]:
%%writefile battle.py
from game_simulator_v2 import Character


class Battle:
    def __init__(self, hero: Character, enemy: Character):
        # Store the two characters in the battle.
        self.hero = hero
        self.enemy = enemy

    def play_round(self):
        # Hero attacks first.
        self.hero.attack(self.enemy)
        # Enemy attacks only if still alive.
        if self.enemy.is_alive:
            self.enemy.attack(self.hero)

    def simulate_battle(self):
        # Repeat rounds until one character is dead.
        round_number = 0
        while self.hero.is_alive and self.enemy.is_alive:
            round_number += 1
            self.play_round()
            print(f"Round {round_number}: {self.hero.name} {self.hero.hp} HP, "
                  f"{self.enemy.name} {self.enemy.hp} HP")

        if self.hero.is_alive:
            winner = self.hero
        else:
            winner = self.enemy
        print("Winner:", winner.name)
        return winner


Overwriting battle.py


### Battle 1 Without weapons



In [20]:
from battle import Battle

# Simulate a battle between two characters without weapons.
hero = Warrior("Knight", hp_max=40, damage=12, armor=4)
enemy = Character("Goblin", hp_max=45, damage=8)

battle1 = Battle(hero, enemy)
battle1.simulate_battle()

Round 1: Knight 36 HP, Goblin 33 HP
Round 2: Knight 32 HP, Goblin 21 HP
Round 3: Knight 28 HP, Goblin 9 HP
Round 4: Knight 28 HP, Goblin 0 HP
Winner: Knight


Character(name=Knight, hp=28, damage=12)

### Battle 2 With weapons



In [21]:
# Task 3 and 5: Equip both characters and simulate another battle.
sword = Weapon("Sword", Dice(12))
staff = Weapon("Staff", Dice(10))

hero = Warrior("Knight", hp_max=60, damage=9, weapon=sword, armor=4)
enemy = Mage("Wizard", hp_max=45, damage=10, weapon=staff)

battle2 = Battle(hero, enemy)
battle2.simulate_battle()

Round 1: Knight 44 HP, Wizard 38 HP
Round 2: Knight 28 HP, Wizard 34 HP
Round 3: Knight 12 HP, Wizard 25 HP
Round 4: Knight 6 HP, Wizard 17 HP
Round 5: Knight 0 HP, Wizard 16 HP
Winner: Wizard


Character(name=Wizard, hp=16, damage=10)

## Simple checks

These small tests check the main rules from Tasks 1–5.

In [22]:
#  HP stays between 0 and hp_max, and is_alive works.
test = Character("Test", hp_max=20)
test.take_damage(100)
assert test.hp == 0 and not test.is_alive
test.heal(100)
assert test.hp == 20 and test.is_alive

#  Dice roll is within range; fewer than 2 sides is not allowed.
assert 1 <= Dice(6).roll() <= 6
assert repr(Dice()) == "Dice(sides=6)"
try:
    Dice(1)
    raise AssertionError("Dice(1) should fail")
except ValueError:
    pass

# A weapon uses its dice for damage.
weapon_test = Weapon("Test sword", Dice(6))
assert 1 <= weapon_test.roll_damage() <= 6

# Warrior reduces damage by armor.
armored = Warrior("Armored", hp_max=20, armor=5)
armored.take_damage(7)
assert armored.hp == 18

# Magic ignores weapons and uses 10 mana for double damage.
magic_target = Character("Dummy", hp_max=30)
magic_user = Mage("Mage", damage=6, weapon=weapon_test)
magic_user.attack(magic_target)
assert magic_target.hp == 18 and magic_user.mana == 20

# With less than 10 mana, a normal attack restores 5 mana.
magic_user.mana = 5
magic_user.attack(magic_target)
assert magic_user.mana == 10

# Enemy cannot attack back after being defeated.
fast_hero = Character("Hero", hp_max=20, damage=20)
fast_enemy = Character("Enemy", hp_max=10, damage=10)
Battle(fast_hero, fast_enemy).play_round()
assert fast_hero.hp == 20 and fast_enemy.hp == 0

print("All checks passed!")

All checks passed!


In [23]:
%%writefile requirements.txt
# Python 3.11 or newer
ipykernel


Overwriting requirements.txt


## GenAI use 
During this assignment, we used AI tools to help us understand parts of the Python code
and improve the presentation.
 The AI tools were useful for explaining
different approaches to solving problems and helping us understand how certain
functions work.
 We also used AI tools to get suggestions for improving our code and
solving some programming errors.
 We reviewed the suggestions and made sure they were relevant to
the assignment
## GitHub Repository
Our GitHub repository: https://github.com/minibergset/git-practice